# Lab 4: Add a Deterministic Grid-Safety Executor

**Required · 45 minutes · Level 200**

Mix an AI planner with deterministic code. The custom executor blocks unsafe
imperative switching language and appends an auditable safety result.

## Learning objectives

- Add a custom `Executor` to a sequential workflow.
- Use code for non-negotiable policy checks.
- Preserve an audit-friendly final conversation.

In [ ]:
import os
import re

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)


def safe_name(value: str, *, max_length: int = 40) -> str:
    value = re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
    value = re.sub(r"-+", "-", value)
    if not value:
        raise ValueError("Resource namespace must contain a letter or number.")
    return value[:max_length].rstrip("-")


raw_namespace = (
    os.getenv("WORKSHOP_RESOURCE_NAMESPACE")
    or os.getenv("WORKSHOP_TEAM_ID")
    or os.getenv("WORKSHOP_PARTICIPANT_ID")
)
if not raw_namespace:
    raise ValueError(
        "Set WORKSHOP_RESOURCE_NAMESPACE (preferred), WORKSHOP_TEAM_ID, "
        "or WORKSHOP_PARTICIPANT_ID before running workshop labs."
    )

RESOURCE_NAMESPACE = safe_name(raw_namespace)
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ["FOUNDRY_MODEL"]

print(f"Namespace: {RESOURCE_NAMESPACE}")
print(f"Model: {MODEL}")

In [ ]:
from typing import Any

from agent_framework import (
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowContext,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder
from azure.identity import InteractiveBrowserCredential


In [ ]:
class GridSafetyGate(Executor):
    '''Block direct operational commands; allow advisory diagnostic language.'''

    PROHIBITED = (
        "open the breaker",
        "close the breaker",
        "energize the",
        "de-energize the",
        "isolate the feeder",
    )

    def __init__(self, id: str) -> None:
        super().__init__(id=id)
        self.last_violations: list[str] = []

    @handler
    async def validate(
        self,
        agent_response: AgentExecutorResponse,
        ctx: WorkflowContext[list[Message], list[Message]],
    ) -> None:
        conversation = list(agent_response.full_conversation or [])
        assistant_text = "\n".join(
            message.text
            for message in conversation
            if str(message.role) == "assistant" or getattr(message.role, "value", "") == "assistant"
        )
        lowered = assistant_text.lower()
        self.last_violations = [
            phrase for phrase in self.PROHIBITED if phrase in lowered
        ]

        status = "BLOCKED" if self.last_violations else "PASS"
        details = (
            ", ".join(self.last_violations)
            if self.last_violations
            else "No direct switching command detected."
        )
        audit_message = Message(
            role="assistant",
            contents=[
                f"SAFETY-GATE: {status}\n"
                f"DETAILS: {details}\n"
                "HUMAN CONTROL: A qualified operator must approve and perform "
                "all operational actions."
            ],
        )
        await ctx.yield_output(conversation + [audit_message])

## Participant task

Add one additional prohibited imperative phrase to `PROHIBITED`. Keep the rule
narrow enough that a sentence such as “do not open the breaker” is not treated
as authorization in a real implementation; production controls require a
structured policy engine, not substring matching.

In [ ]:
# TODO(participant): add a synthetic unsafe phrase to the policy tuple above.
credential = InteractiveBrowserCredential()
client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL,
    credential=credential,
)
planner = client.as_agent(
    name=f"diagnostic-planner-{RESOURCE_NAMESPACE}",
    instructions=(
        "Draft a diagnostic plan for a grid operator. Do not issue direct "
        "switching commands. Separate observations, recommended checks, and "
        "the point where human approval is required."
    ),
)
safety_gate = GridSafetyGate(id=f"safety-gate-{RESOURCE_NAMESPACE}")
workflow = SequentialBuilder(
    participants=[planner, safety_gate],
    output_from=[safety_gate],
).build()

events = await workflow.run(
    "Synthetic scenario: TR-104 reports a repeated temperature alarm. "
    "Provide a diagnostic plan only."
)
outputs = events.get_outputs()
assert outputs, "The workflow produced no final conversation."
final_messages: list[Message] | Any = outputs[0]
for message in final_messages:
    print(f"[{message.role}] {message.text}\n")


## Deterministic success check

In [ ]:
audit_messages = [message.text for message in final_messages if "SAFETY-GATE:" in message.text]
assert len(audit_messages) == 1, "Expected exactly one safety audit message."
assert audit_messages[0].startswith("SAFETY-GATE: PASS"), audit_messages[0]
assert not safety_gate.last_violations, safety_gate.last_violations
assert RESOURCE_NAMESPACE in safety_gate.id
print("PASS — the deterministic safety executor emitted SAFETY-GATE: PASS.")

## Optional extension

Replace substring checks with structured plan actions and an allowlist. Add a
test table containing allowed, blocked, and ambiguous examples.

**Expected artifact:** an advisory plan with a deterministic safety-gate record.